# 03 — Prepare Contextual SFT Data and Freeze the Baseline Benchmark

This notebook consumes the outputs of **02_BUILD_CHILD_CAREGIVER_DATASET.ipynb** and prepares the
data for the first Base → LoRA experiment.

It creates two distinct datasets:

1. **Language SFT** — predicts a real child utterance and excludes `<SILENCE>` targets.
2. **Interaction SFT** — retains child `<SILENCE>` targets for a later turn-taking experiment.

It also creates **multi-turn contextual examples**, checks child leakage and age/length distributions,
and freezes a test benchmark that must remain unchanged across Base, LoRA, and later QLoRA runs.

### Important experimental rule

Do **not** tune prompts, generation settings, or model-selection decisions on the frozen test set.
Use training data to fit the model and validation data for decisions. Evaluate the frozen test set
only with the finalized configuration.


In [ ]:
from pathlib import Path
import json
import hashlib
import re

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / "data"
DATA_DIR = DATA_ROOT / "processed" / "gleason"
OUTPUT_DIR = DATA_ROOT / "processed" / "sft_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TURNS_FILE = DATA_DIR / "gleason_turns_age4_6.csv"
PAIRS_FILE = DATA_DIR / "gleason_interaction_pairs_age4_6.csv"
SPLIT_FILE = DATA_DIR / "gleason_child_split_manifest.csv"

# Number of preceding real turns included before the caregiver prompt.
MAX_CONTEXT_TURNS = 6

# Frozen benchmark size. None = keep every eligible test example.
MAX_FROZEN_TEST_EXAMPLES = None

RANDOM_STATE = 42


## 1. Load and validate the outputs from Notebook 02


In [ ]:
turns = pd.read_csv(TURNS_FILE)
pairs = pd.read_csv(PAIRS_FILE)
manifest = pd.read_csv(SPLIT_FILE)

required_turn_cols = {
    "conversation_id", "segment_id", "child_id", "age_months",
    "source_turn_index", "role", "text", "split"
}
required_pair_cols = {
    "conversation_id", "segment_id", "child_id", "age_months",
    "prompt_role", "prompt", "response_role", "response",
    "response_is_silence", "split"
}

assert required_turn_cols.issubset(turns.columns)
assert required_pair_cols.issubset(pairs.columns)

print(f"Turns: {len(turns):,}")
print(f"Interaction pairs: {len(pairs):,}")
print(f"Unique children: {turns['child_id'].nunique()}")
display(
    turns.groupby("split").agg(
        children=("child_id", "nunique"),
        conversations=("conversation_id", "nunique"),
        turns=("text", "size"),
    )
)


## 2. Audit `<SILENCE>`

The first LoRA experiment is a **language-generation** experiment, so its target must be an observed
child utterance. `<SILENCE>` remains available in a separate interaction dataset but is not used as
a target in Language SFT.


In [ ]:
child_response_pairs = pairs[
    (pairs["prompt_role"] == "caregiver") &
    (pairs["response_role"] == "child")
].copy()

silence_summary = (
    child_response_pairs.groupby("split")
    .agg(
        child_targets=("response", "size"),
        silence_targets=("response_is_silence", "sum")
    )
)
silence_summary["silence_rate"] = (
    silence_summary["silence_targets"] / silence_summary["child_targets"]
)
display(silence_summary.round(3))

language_pairs = child_response_pairs[
    ~child_response_pairs["response_is_silence"].astype(bool)
].copy()

interaction_pairs = child_response_pairs.copy()

print(f"Language targets (real child speech): {len(language_pairs):,}")
print(f"Interaction targets (speech + silence): {len(interaction_pairs):,}")


## 3. Build contextual child-response examples

Instead of training only on one caregiver utterance, each example uses up to the previous
`MAX_CONTEXT_TURNS` **real source turns within the same uninterrupted interaction segment**.

The target is a real child turn whose immediately preceding real turn is a caregiver turn.
This prevents context from crossing an excluded-investigator boundary and avoids manufacturing
direct caregiver–child exchanges that did not occur in the source transcript.


In [ ]:
def speaker_label(role):
    return "Child" if role == "child" else "Caregiver"


def make_context_examples(turns_df, max_context_turns=6):
    examples = []

    group_cols = ["conversation_id", "segment_id"]

    for (conversation_id, segment_id), group in turns_df.groupby(group_cols, sort=False):
        g = group.sort_values("source_turn_index").reset_index(drop=True)

        for i in range(1, len(g)):
            target = g.iloc[i]
            previous = g.iloc[i - 1]

            # Language SFT: actual child response directly following caregiver speech.
            if target["role"] != "child" or previous["role"] != "caregiver":
                continue

            start = max(0, i - max_context_turns)
            history = g.iloc[start:i]

            context_lines = [
                f"{speaker_label(row['role'])}: {row['text']}"
                for _, row in history.iterrows()
            ]

            examples.append({
                "example_id": f"{conversation_id}::seg{segment_id}::turn{int(target['source_turn_index'])}",
                "corpus": target.get("corpus", "Gleason"),
                "condition": target.get("condition"),
                "conversation_id": conversation_id,
                "segment_id": int(segment_id),
                "child_id": target["child_id"],
                "age_months": float(target["age_months"]),
                "split": target["split"],
                "context_turn_count": len(history),
                "context": "\n".join(context_lines),
                "target": target["text"],
                "target_source_turn_index": int(target["source_turn_index"]),
            })

    return pd.DataFrame(examples)


context_df = make_context_examples(turns, MAX_CONTEXT_TURNS)

print(f"Contextual language examples: {len(context_df):,}")
display(context_df.head(5))
display(context_df.groupby("split").size().rename("examples").to_frame())


## 4. Format training records

The JSONL format below is deliberately model-agnostic. A later training notebook can map these
records to the exact chat template required by Qwen, SmolLM, Llama, or another base model.

The instruction conditions on the child's age in months but does not tell the model to imitate a
specific named child.


In [ ]:
def format_language_record(row):
    instruction = (
        f"Simulate the verbal response of a {row.age_months:.1f}-month-old child "
        "in a natural caregiver-child conversation. "
        "Respond as the child only; do not add a speaker label."
    )

    return {
        "id": row.example_id,
        "instruction": instruction,
        "input": row.context + "\nChild:",
        "output": row.target,
        "age_months": row.age_months,
        "child_id": row.child_id,
        "conversation_id": row.conversation_id,
        "condition": row.condition,
        "context_turn_count": int(row.context_turn_count),
        "split": row.split,
    }


language_records = [
    format_language_record(row)
    for row in context_df.itertuples(index=False)
]

language_records[:2]


## 5. Build the separate Interaction SFT dataset

This preserves the simpler caregiver → child interaction pairs from Notebook 02, including
`<SILENCE>`. Keep it separate from Language SFT so the first experiment can answer a clean question:
**does LoRA improve child-language generation?**

A later experiment can explicitly test whether learning turn-taking/silence improves dynamic
interaction.


In [ ]:
interaction_records = []

for row in interaction_pairs.itertuples(index=False):
    interaction_records.append({
        "instruction": (
            f"Simulate the next behavior of a {row.age_months:.1f}-month-old child "
            "in a caregiver-child interaction. Respond with the child's utterance, "
            "or <SILENCE> when the child does not respond."
        ),
        "input": f"Caregiver: {row.prompt}\nChild:",
        "output": row.response,
        "age_months": float(row.age_months),
        "child_id": row.child_id,
        "conversation_id": row.conversation_id,
        "condition": row.condition,
        "split": row.split,
        "is_silence": bool(row.response_is_silence),
    })

print(f"Interaction SFT records: {len(interaction_records):,}")


## 6. Leakage checks


In [ ]:
children_by_split = {
    split: set(group["child_id"])
    for split, group in context_df.groupby("split")
}

assert children_by_split["train"].isdisjoint(children_by_split["validation"])
assert children_by_split["train"].isdisjoint(children_by_split["test"])
assert children_by_split["validation"].isdisjoint(children_by_split["test"])

# Conversation IDs should also remain split-exclusive.
conv_by_split = {
    split: set(group["conversation_id"])
    for split, group in context_df.groupby("split")
}
assert conv_by_split["train"].isdisjoint(conv_by_split["validation"])
assert conv_by_split["train"].isdisjoint(conv_by_split["test"])
assert conv_by_split["validation"].isdisjoint(conv_by_split["test"])

print("PASS: no child or conversation leakage across splits.")
display(
    context_df.groupby("split").agg(
        children=("child_id", "nunique"),
        conversations=("conversation_id", "nunique"),
        examples=("example_id", "size"),
        mean_age_months=("age_months", "mean"),
        min_age_months=("age_months", "min"),
        max_age_months=("age_months", "max"),
    ).round(2)
)


## 7. Length and age diagnostics

These are simple whitespace-token diagnostics, not model-tokenizer counts. Exact tokenizer lengths
should be computed after the base model is chosen.


In [ ]:
context_df["target_words"] = context_df["target"].fillna("").str.split().str.len()
context_df["input_words"] = context_df["context"].fillna("").str.split().str.len()

display(
    context_df.groupby("split").agg(
        examples=("example_id", "size"),
        mean_input_words=("input_words", "mean"),
        median_input_words=("input_words", "median"),
        p95_input_words=("input_words", lambda x: x.quantile(.95)),
        mean_target_words=("target_words", "mean"),
        median_target_words=("target_words", "median"),
        p95_target_words=("target_words", lambda x: x.quantile(.95)),
    ).round(2)
)


In [ ]:
plt.figure(figsize=(8, 5))
for split, group in context_df.groupby("split"):
    plt.hist(group["target_words"], bins=30, alpha=0.5, label=split)
plt.xlabel("Child target length (whitespace-separated words)")
plt.ylabel("Examples")
plt.title("Child-response length distribution")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
for split, group in context_df.groupby("split"):
    plt.hist(group["age_months"], bins=14, alpha=0.5, label=split)
plt.xlabel("Age (months)")
plt.ylabel("Examples")
plt.title("Age distribution by split")
plt.legend()
plt.tight_layout()
plt.show()


## 8. Freeze the baseline test benchmark

The benchmark contains only held-out test-child examples. It is shuffled once with a fixed seed,
assigned stable benchmark IDs, and written to disk.

A SHA-256 checksum is also saved. Use the same file for every Base/LoRA/QLoRA comparison.


In [ ]:
frozen = (
    context_df[context_df["split"] == "test"]
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

if MAX_FROZEN_TEST_EXAMPLES is not None:
    frozen = frozen.head(MAX_FROZEN_TEST_EXAMPLES).copy()

frozen["benchmark_id"] = [f"GLEASON_TEST_{i:05d}" for i in range(len(frozen))]

frozen_records = []
for row in frozen.itertuples(index=False):
    rec = format_language_record(row)
    rec["benchmark_id"] = row.benchmark_id
    # Reference output is retained for automatic evaluation, but should not be shown to the model.
    rec["reference"] = rec.pop("output")
    frozen_records.append(rec)

FROZEN_PATH = OUTPUT_DIR / "gleason_frozen_baseline_test.jsonl"
with FROZEN_PATH.open("w", encoding="utf-8") as f:
    for rec in frozen_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

checksum = hashlib.sha256(FROZEN_PATH.read_bytes()).hexdigest()
(OUTPUT_DIR / "gleason_frozen_baseline_test.sha256").write_text(
    checksum + "  " + FROZEN_PATH.name + "\n",
    encoding="utf-8"
)

print(f"Frozen test examples: {len(frozen_records):,}")
print("SHA-256:", checksum)
display(pd.DataFrame(frozen_records).head(3))


## 9. Export Hugging Face-compatible JSONL files

The first LoRA run should use:

- `language_sft_train.jsonl`
- `language_sft_validation.jsonl`

Do **not** train on `language_sft_test.jsonl` or `gleason_frozen_baseline_test.jsonl`.


In [ ]:
def write_jsonl(path, records):
    with Path(path).open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


for split in ["train", "validation", "test"]:
    split_records = [r for r in language_records if r["split"] == split]
    write_jsonl(OUTPUT_DIR / f"language_sft_{split}.jsonl", split_records)

    interaction_split = [r for r in interaction_records if r["split"] == split]
    write_jsonl(
        OUTPUT_DIR / f"interaction_sft_{split}.jsonl",
        interaction_split
    )

context_df.to_csv(OUTPUT_DIR / "contextual_language_examples.csv", index=False)

print("Created:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)


## 10. Save an experiment manifest

This records the preprocessing decisions that must accompany your experimental results.


In [ ]:
experiment_manifest = {
    "corpus": "Gleason CHILDES",
    "age_range_months": {"minimum_inclusive": 48, "maximum_exclusive": 72},
    "split_unit": "child_id",
    "context_turns_max": MAX_CONTEXT_TURNS,
    "language_sft_silence_targets": "excluded",
    "interaction_sft_silence_targets": "retained",
    "frozen_test_examples": len(frozen_records),
    "frozen_test_sha256": checksum,
    "random_state": RANDOM_STATE,
    "notes": [
        "Language SFT uses real child utterances directly following caregiver turns.",
        "Context never crosses an interaction segment boundary created by an excluded speaker.",
        "Test references must never be included in model prompts.",
        "Model-specific token counts must be computed after choosing the base tokenizer."
    ],
}

with (OUTPUT_DIR / "experiment_manifest.json").open("w", encoding="utf-8") as f:
    json.dump(experiment_manifest, f, indent=2)

display(pd.DataFrame([{
    "train_examples": sum(r["split"] == "train" for r in language_records),
    "validation_examples": sum(r["split"] == "validation" for r in language_records),
    "test_examples": sum(r["split"] == "test" for r in language_records),
    "interaction_examples": len(interaction_records),
    "frozen_test_examples": len(frozen_records),
}]))


# Next experiment: Base-model benchmark

After this notebook:

1. Choose **one base model**.
2. Freeze its exact chat template and generation settings.
3. Generate responses for every record in `gleason_frozen_baseline_test.jsonl`.
4. Save model outputs with `benchmark_id`.
5. Compute baseline child-language metrics.
6. Train LoRA on `language_sft_train.jsonl`, using only `language_sft_validation.jsonl` for tuning.
7. Run the fine-tuned model on the **same frozen benchmark with the same generation settings**.
8. Compare Base vs. LoRA.
9. Only after that experiment is complete, introduce `<SILENCE>` learning or preference optimization.

This keeps the first experiment interpretable: the primary changed variable is supervised LoRA
fine-tuning on natural child responses.
